In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

# ─────────────────────────────────────────────
# 1. 크롬 드라이버 설정
# ─────────────────────────────────────────────
service = Service(ChromeDriverManager().install())

options = Options()
options.add_argument("--window-size=1920x1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")

driver = webdriver.Chrome(service=service, options=options)

base_url = "https://books.toscrape.com/"
driver.get(base_url)

wait = WebDriverWait(driver, 10)

# ─────────────────────────────────────────────
# 2. 평점 매핑
# ─────────────────────────────────────────────
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# ─────────────────────────────────────────────
# 3. 카테고리 버튼 클릭 방식 크롤링
# ─────────────────────────────────────────────
# 카테고리 버튼들 가져오기
categories = driver.find_elements(By.CSS_SELECTOR, "div.side_categories ul li ul li a")
print(f"총 {len(categories)}개 카테고리 발견 ✅")

# 모든 카테고리를 순서대로 탐색
for cat_idx in range(len(categories)):
    # 매번 다시 찾아야 클릭 가능 (페이지 이동 후 DOM이 새로 로드됨)
    categories = driver.find_elements(By.CSS_SELECTOR, "div.side_categories ul li ul li a")
    cat = categories[cat_idx]

    cat_name = cat.text.strip()
    print(f"\n\n===== {cat_idx+1}. 카테고리 클릭: {cat_name} =====")

    # 버튼 클릭
    cat.click()
    time.sleep(2)

    # ── 카테고리 내부 페이지 크롤링 ──
    page_num = 1
    while True:
        print(f"\n--- {cat_name} {page_num} 페이지 ---")

        # 페이지 내 책 정보 수집
        titles = driver.find_elements(By.CSS_SELECTOR, "article.product_pod > h3")
        price_gbps = driver.find_elements(By.CSS_SELECTOR, "p.price_color")
        stocks = driver.find_elements(By.CSS_SELECTOR, "p.instock.availability")
        ratings = driver.find_elements(By.CSS_SELECTOR, "p.star-rating")
        product_links = driver.find_elements(By.CSS_SELECTOR, "article.product_pod h3 a")

        for i in range(len(titles)):
            # 평점 텍스트 → 숫자 변환
            rating_text = [c for c in ratings[i].get_attribute("class").split() if c != "star-rating"][0] if i < len(ratings) else None
            rating_number = rating_map.get(rating_text, 0) if rating_text else None

            record = {
                "category": cat_name,
                "title": titles[i].text if i < len(titles) else None,
                "price": price_gbps[i].text.replace("£","") if i < len(price_gbps) else None,
                "stock": stocks[i].text if i < len(stocks) else None,
                "rating": rating_number,
                "url": product_links[i].get_attribute("href") if i < len(product_links) else None,
                "page": page_num
            }
            print(record)

        # 다음 페이지 버튼 확인
        try:
            next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next > a")))
            next_btn.click()
            time.sleep(2)
            page_num += 1
        except:
            print(f"👉 {cat_name} 카테고리 마지막 페이지 도달")
            break

    # 다시 홈으로 돌아가야 다음 카테고리 클릭 가능
    driver.get(base_url)
    time.sleep(1)

# ─────────────────────────────────────────────
# 4. 종료
# ─────────────────────────────────────────────
driver.quit()